we have 3 files to use- adt raw csv
fate readout csv at barcode lvl 
x rna counts mtx 

subset to hsc . find hvgs . 
use the fate readout csv to classify in terms of fate what barcode lineages are unipotent 
broadcast those lineages onto every cell in the barcode lineage
this is the basis of our groups = True in mofa. We will run with some larger groups first- ery / Mast [largest presence iirc] / cDC1 VS multipotent . see wht makes these JUST ery and not multipotent. so run w groups=True for group= i for all i in celltypes . 

to check if this has worked, check the variance explained per factor in the data. 

if it doesnt work: try run on all data and not just hscs 

In [1]:
import pandas as pd
import scanpy as sc 
import numpy as np 

In [2]:
adt_counts = pd.read_csv("/vast/projects/Sisseq/givanna/making_sense_of_files/adt_raw.csv")

print(adt_counts.head())
print(adt_counts.columns.to_list())

# important here -> protein is column , cell is row so (cell x protien)



   Unnamed: 0                cell  Hu.CD134  Hu.CD112  Hu.HLA.DR  Hu.CD115  \
0           1  AAACCCAAGGTTAAAC-1         0        25         19         1   
1           2  AAACCCACAAATCAAG-1         0         8         18         3   
2           3  AAACCCACAAGTATCC-1         0         8          7         0   
3           4  AAACCCACAGGTTACT-1         0         9         13         2   
4           5  AAACCCACATGATCTG-1         0        14          1        13   

   Hu.CD279  Hu.CD37  Hu.CD99  Hu.LOX.1  ...  RNA.weight  ADT.weight  \
0         0        0        9         0  ...    0.529892    0.470108   
1         0        0       16         0  ...    0.603510    0.396490   
2         1        0        5         1  ...    0.657347    0.342653   
3         0        0       13         2  ...    0.423450    0.576550   
4         3        0        3         1  ...    0.525541    0.474459   

   wsnn_res.2  seurat_clusters  wsnn_res.1  predicted.celltype.score  \
0          10             

In [3]:
num_barcodes = adt_counts["cons_bc"].nunique()

print("number of unique barcodes is : ", num_barcodes)

adt_counts["cons_bc"].head() # length of 20. the subsequent analysis has the length to be subset to 15 and donor id to be added here for analysis

number of unique barcodes is :  3066


0    GTCTCGTTCTGGGTGGTAGT
1                     NaN
2                     NaN
3    CCCTGGTTGTCTGTGTTTGT
4    AGGAGGGACATTCAGGGACT
Name: cons_bc, dtype: object

In [4]:
adt_obj = adt_counts

adt_obj["cons_bc"] = adt_counts["cons_bc"].str[:15]
adt_obj["barcode_donor"] = adt_obj["cons_bc"].str.cat(adt_counts['donor_id'], sep='_')

len(adt_obj["barcode_donor"])
adt_obj["barcode_donor"].head(30)

# no. of barcodes after subsetting and attaching donor - 9768
# lots of NA values. try dropNA ? 

adt_obj["barcode_donor"].isna().sum() # 4259 are NA 
adt_obj = adt_obj[adt_obj["barcode_donor"].notna()]
adt_obj["barcode_donor"].isna().sum()
len(adt_obj["barcode_donor"]) # 5509 barcodes remaining

5509

In [5]:
# subsetting adt_obj to hscs 
print(adt_obj['populations'].head())
adt_obj = adt_obj[adt_obj["populations"] == "HSC"]
len(adt_obj)

# no. left are now 439 

0                           Mk
3                           Mk
4    early_lymphoid_progenitor
6                           Mk
7                  Mk-ery_prog
Name: populations, dtype: object


439

In [6]:
# now should have the same barcodes for fate and X_rna 

fate_obj = pd.read_csv("/vast/projects/Sisseq/givanna/making_sense_of_files/fate_readout.csv")
fate_obj.columns # (barcode x day_PX for X in A,B,C)
fate_obj.head() # we have barcode_first_15_bp 

# tryig to overlap the barcode_donor col of the prev dataset with the bcode_first_15bp of this dataset 

overlap = set(adt_obj["barcode_donor"]) & set(fate_obj["bcode_first_15bp"])

print(f"Number of overlapping barcodes: {len(overlap)}")
print(list(overlap)[:20])

print(fate_obj["bcode_first_15bp"].head())
adt_obj["barcode_donor"].head()

# we are stuck at 112 again. Number of overlapping barcodes: 112

# abandon this and work on the 439 for now. 


Number of overlapping barcodes: 112
['TACAGGCTGTACGTC_P', 'ACCTGCGTGTATGTC_1', 'CTGCAAGGTTGCTAT_U', 'TCGAGATTCATCCTC_1', 'TGGAGTCTCTGGCTC_P', 'AGGTGTTTGTGTCTC_1', 'ATCAGGTTGACCGAG_1', 'TAGAGTTAGAACGAG_1', 'TTCTGAGAGTCGCTG_1', 'TAGCAATCTTTCAAA_U', 'AATGAATCTATGGAG_U', 'CACTCGATGTGTGTC_1', 'CCCAGTGACATAGTG_1', 'GTCCAAGCCACGATT_U', 'CTTGGTACATTGTTT_U', 'GATGGTAGTACGTAC_U', 'CGGTGGTTCTCTCTC_1', 'TTTCTATGAGAAACA_V', 'TTGTCGCTGTGCCTC_P', 'GAGTGGCTCTGTCAG_1']
0    AAACAAACGAGGGAT_U
1    AAACAAACTTGGAAT_U
2    AAACAAAGTAGGCAG_U
3    AAACAAGCGTAGTAG_U
4    AAACAAGGCTCGGAG_U
Name: bcode_first_15bp, dtype: object


28     CTGCCTGGCTTCGTA_1
86     ACCTGGGTGAGWGTG_1
87     TTGACCAAGATTGTG_1
150    TTCTGAGAGTCGCTG_1
151    AACTGACACACAGTG_1
Name: barcode_donor, dtype: object

In [7]:
# reading rna counts scanpy

rna_counts = sc.read_mtx("/vast/projects/Sisseq/givanna/making_sense_of_files/X_rna_counts.mtx") # of type andata 

print(rna_counts)
# just intersect w adt_obj. 

AnnData object with n_obs × n_vars = 9768 × 36601


In [10]:
import anndata as ad
from scipy.sparse import csr_matrix

In [11]:
rna_counts.X

<9768x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 44068873 stored elements in Compressed Sparse Row format>

In [15]:
rna_counts.obsnames = [f"Cell_{i:d}" for i in range(rna_counts.n_obs)]
rna_counts.obs_names[:10]

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')

## Step 1: Inspect fate_readout.csv columns

Before we can compute per-barcode dominance, we need to know exactly how the
lineage x day x plate columns are named in `fate_readout.csv`. Run this once,
eyeball the printed column list, then fix `LINEAGE_COLS` in the next cell to
match. We already know from `main.R` that columns are not simply `day_celltype`
— there's also plate (PA/PB/PC) baked in, so we sum across plates per
day/lineage before scoring.

In [ ]:
# fate_obj is already loaded above (cell 6) from fate_readout.csv
# just re-print columns here so this section is self-contained if run standalone
print(fate_obj.shape)
print(fate_obj.columns.tolist())

## Step 2: Classify each barcode as unipotent (+ dominant lineage) or multipotent

For every barcode, sum its readout across all days *and* all plates for each
of the 8 lineages, then compute `dominance` = (top lineage total) / (total
across all lineages). If `dominance >= threshold`, the barcode is labelled
with its dominant lineage name (e.g. `"Ery"`); otherwise it's `"multipotent"`.

This gives one categorical label per barcode — mutually exclusive by
construction, so it's safe to feed straight into MOFA's `groups=True`.

**You need to edit `LINEAGE_COLS` below** to match the actual column names
printed in Step 1. The structure is a dict: lineage name -> list of all
columns (across days/plates) that belong to that lineage. Example shape
(placeholder — replace with real column names):
```python
LINEAGE_COLS = {
    "Ery":  ["D7_Ery_PA", "D7_Ery_PB", "D7_Ery_PC", "D10_Ery_PA", ...],
    "Mast": [...],
    ...
}
```

In [ ]:
import re

# ==== EDIT THIS: map each of the 8 lineages to its columns in fate_obj ====
# quick-start: if columns follow a consistent "<lineage>" substring pattern,
# this regex-based auto-grouping may work out of the box. Inspect the printed
# groups below and correct LINEAGE_COLS manually if any column lands wrong.
LINEAGE_NAMES = ["Ery", "Lymph", "Mast", "Mye", "Rest", "cDC1", "cDC2", "pDC"]

LINEAGE_COLS = {lin: [] for lin in LINEAGE_NAMES}
unmatched_cols = []

for col in fate_obj.columns:
    matched = False
    for lin in LINEAGE_NAMES:
        # case-insensitive substring match on lineage name within column name
        if re.search(lin, col, flags=re.IGNORECASE):
            LINEAGE_COLS[lin].append(col)
            matched = True
            break
    if not matched:
        unmatched_cols.append(col)

print("Auto-grouped column counts per lineage:")
for lin, cols in LINEAGE_COLS.items():
    print(f"  {lin}: {len(cols)} cols")
print("\nColumns that did NOT match any lineage (likely metadata, e.g. patient/plate id — fine to ignore, but check):")
print(unmatched_cols)

In [ ]:
def classify_unipotency(fate_df, lineage_cols, dominance_threshold=0.7):
    """
    Score each barcode (row) for lineage dominance and assign a group label.

    fate_df        : one row per barcode (index = barcode identifier used for
                      the later merge, e.g. bcode_first_15bp + donor)
    lineage_cols    : dict {lineage_name: [columns belonging to that lineage]},
                      summed across day + plate
    dominance_threshold : fraction of total lineage signal the top lineage must
                      hold for a barcode to be called unipotent. 0.7 is a
                      reasonable starting point -- validate against the manual
                      cluster calls (clusters 8 & 17 -> multipotent; clusters
                      4/5/9/12/13/16 -> unipotent) and tune if needed.

    Returns a dataframe indexed like fate_df with:
      dominant_lineage : name of the top lineage regardless of threshold
      dominance        : top lineage total / total across all lineages
      group_label      : dominant_lineage if unipotent, else "multipotent"
                          -> this is the column that goes into MOFA groups=True
    """
    lineage_totals = pd.DataFrame(
        {lin: fate_df[cols].clip(lower=0).sum(axis=1) for lin, cols in lineage_cols.items() if len(cols) > 0},
        index=fate_df.index,
    )

    total = lineage_totals.sum(axis=1)
    dominant_lineage = lineage_totals.idxmax(axis=1)
    dominance = lineage_totals.max(axis=1) / total.replace(0, np.nan)

    is_unipotent = dominance >= dominance_threshold
    group_label = np.where(is_unipotent, dominant_lineage, "multipotent")

    out = pd.DataFrame({
        "dominant_lineage": dominant_lineage,
        "dominance": dominance,
        "group_label": group_label,
    }, index=fate_df.index)

    # barcodes with zero total signal across all lineages can't be scored
    out.loc[total == 0, ["dominant_lineage", "dominance", "group_label"]] = np.nan
    return out


# fate_obj is indexed by row (barcode), keyed via bcode_first_15bp elsewhere.
# use bcode_first_15bp as the merge key going forward, consistent with the R
# reconciliation work (full-length cons_bc join retains fewer clones, see
# definitive_missing_barcode_proof.R).
fate_indexed = fate_obj.set_index("bcode_first_15bp")
barcode_labels = classify_unipotency(fate_indexed, LINEAGE_COLS, dominance_threshold=0.7)

print(barcode_labels["group_label"].value_counts(dropna=False))
barcode_labels.head()

In [ ]:
# Sanity check against the manually-labelled clusters from the plot:
# clusters 8 & 17 should skew multipotent; 4/5/9/12/13/16 should skew unipotent.
# If you have a barcode -> cluster mapping dataframe (e.g. `barcode_cluster_df`
# with columns ["barcode", "cluster"]), uncomment and run this cross-tab:

# pd.crosstab(barcode_cluster_df["cluster"], barcode_labels.reindex(barcode_cluster_df["barcode"])["group_label"].values)

## Step 3: Preprocess ADT and RNA

- **ADT**: CLR (centered log-ratio) normalisation is the standard for CITE-seq
  protein counts (rather than plain log1p) since it handles the compositional,
  low-dimensionality nature of ADT panels better. Done per-cell, across
  proteins.
- **RNA**: standard scanpy flow — CPM-normalise (`target_sum=1e4` counts-per-
  10k, or `1e6` for true CPM), log1p, then subset to highly variable genes
  (HVGs) so MOFA isn't swamped by noise genes. Then scale (zero mean, unit
  variance, clipped) since MOFA assumes roughly Gaussian-like continuous
  inputs per view.

In [ ]:
# ---- ADT: CLR normalisation ----
adt_feature_cols = [c for c in adt_counts.columns if c.startswith(("Hu.", "HuMs", "Isotype"))]

def clr_normalize(mat):
    """Centered log-ratio, per cell (row), standard for CITE-seq ADT counts."""
    mat = mat.astype(float)
    mat_pseudo = mat + 1  # pseudocount to handle zeros
    log_mat = np.log(mat_pseudo)
    geometric_mean = log_mat.mean(axis=1)
    return log_mat.sub(geometric_mean, axis=0)

adt_clr = clr_normalize(adt_counts[adt_feature_cols])
adt_clr["cell"] = adt_counts["cell"].values
adt_clr["cons_bc"] = adt_counts["cons_bc"].str[:15].values
adt_clr["donor_id"] = adt_counts["donor_id"].values
adt_clr["barcode_donor"] = adt_clr["cons_bc"].str.cat(adt_clr["donor_id"], sep="_")
adt_clr["populations"] = adt_counts["populations"].values

adt_clr.head()

In [ ]:
# ---- RNA: CPM normalise, log1p, HVG subset, scale ----
# rna_counts loaded above via sc.read_mtx; rows = cells (as exported by
# writeMM(t(...)) in the R script), cols = genes.

import os

# NOTE: the R export script (main.R) has a bug -- the writeMM() call for the
# .mtx uses file.path(out_dir, ...) but the two writeLines() calls right after
# it don't, so gene_names/cell_ids can land one directory up from the .mtx
# (in the R working directory at time of export) rather than alongside it.
# Check both candidate locations and use whichever actually matches the
# gene/cell counts in rna_counts, rather than assuming a fixed path.
CANDIDATE_DIRS = [
    "/vast/projects/Sisseq/givanna/making_sense_of_files",
    "/vast/projects/Sisseq/givanna/code",
    "/vast/projects/Sisseq/givanna",
]

def load_names_checked(filename, expected_len):
    for d in CANDIDATE_DIRS:
        path = os.path.join(d, filename)
        if not os.path.exists(path):
            continue
        names = open(path).read().splitlines()
        if len(names) == expected_len:
            print(f"OK: {path} has {len(names)} lines, matches expected {expected_len}")
            return names
        else:
            print(f"SKIP: {path} has {len(names)} lines, expected {expected_len} -- stale/wrong file, not using it")
    raise FileNotFoundError(
        f"Could not find a {filename} with {expected_len} lines in any of {CANDIDATE_DIRS}. "
        f"Likely the R export never wrote a correct copy at any known path -- re-run the export "
        f"with writeLines(..., file.path(out_dir, '{filename}')) (missing file.path() is the bug) "
        f"and point CANDIDATE_DIRS at wherever it lands."
    )

gene_names = load_names_checked("X_rna_counts_gene_names.txt", rna_counts.n_vars)
cell_ids = load_names_checked("X_rna_counts_cell_ids.txt", rna_counts.n_obs)

rna_counts.var_names = gene_names
rna_counts.obs_names = cell_ids
rna_counts.var_names_make_unique()

sc.pp.normalize_total(rna_counts, target_sum=1e4)
sc.pp.log1p(rna_counts)

sc.pp.highly_variable_genes(rna_counts, n_top_genes=2000, flavor="seurat")
rna_hvg = rna_counts[:, rna_counts.var["highly_variable"]].copy()

sc.pp.scale(rna_hvg, max_value=10)

rna_hvg

## Step 4: Broadcast barcode-level group label onto every daughter cell

Every profiled cell carries `cons_bc` + `donor_id` in its metadata. We map
each cell's `barcode_donor` key through `barcode_labels["group_label"]` and
drop cells whose barcode has no fate readout at all (can't be assigned to any
group).

In [ ]:
group_map = barcode_labels["group_label"].to_dict()

adt_clr["group"] = adt_clr["barcode_donor"].map(group_map)

n_before = len(adt_clr)
adt_clr = adt_clr[adt_clr["group"].notna()].copy()
n_after = len(adt_clr)
print(f"ADT cells: {n_before} -> {n_after} after dropping cells with no fate-matched barcode")

print(adt_clr["group"].value_counts())

## Step 5: Build the multi-view, multi-group MOFA input and run

Two views (RNA, ADT), groups = the categorical `group` label (dominant
lineage name, or `"multipotent"`) from Step 4. Cells that don't overlap
between RNA and ADT (matched on `cell` id) are dropped — MOFA needs the same
sample set present across views within a group (missing entries are allowed
and handled internally, but we still need consistent sample IDs).

This uses `mofapy2`'s `entry_point` API directly so we can pass `groups`
explicitly as long-format data, which is the simplest way to hand it
per-cell group assignments.

In [ ]:
# pip install mofapy2 --break-system-packages   # if not already installed
from mofapy2.run.entry_point import entry_point

# align RNA (anndata, obs_names = cell ids) with ADT (dataframe, 'cell' col)
rna_df = pd.DataFrame(
    rna_hvg.X if not hasattr(rna_hvg.X, "toarray") else rna_hvg.X.toarray(),
    index=rna_hvg.obs_names,
    columns=rna_hvg.var_names,
)

common_cells = sorted(set(rna_df.index) & set(adt_clr["cell"]))
print(f"Cells common to RNA and ADT (post group-filter): {len(common_cells)}")

rna_df = rna_df.loc[common_cells]
adt_indexed = adt_clr.set_index("cell").loc[common_cells]
cell_groups = adt_indexed["group"]  # one group label per cell, aligned to common_cells

# ---- build long-format tables mofapy2 expects: sample, group, feature, value, view ----
def to_long(df, view_name, groups):
    long = df.stack().reset_index()
    long.columns = ["sample", "feature", "value"]
    long["view"] = view_name
    long["group"] = long["sample"].map(groups)
    return long

rna_long = to_long(rna_df, "RNA", cell_groups)
adt_long = to_long(adt_indexed[adt_feature_cols], "ADT", cell_groups)

long_data = pd.concat([rna_long, adt_long], ignore_index=True)
long_data.head()

In [ ]:
ent = entry_point()

ent.set_data_options(
    scale_groups=False,
    scale_views=True,   # important with RNA (2000 features) + ADT (~180) at very different scales
)

ent.set_data_df(long_data)

ent.set_model_options(
    factors=15,
    spikeslab_weights=True,
    ard_weights=True,
)

ent.set_train_options(
    iter=1000,
    convergence_mode="fast",
    dropR2=0.001,
    gpu_mode=False,
    seed=42,
)

ent.build()
ent.run()

ent.save("/home/claude/mofa_unipotency_groups.hdf5")

## Step 6: Variance explained per factor per group

This is the actual check for whether the grouping did anything: if certain
factors explain variance strongly in one group (e.g. `Ery`) but not others,
those factors are candidates for what distinguishes that lineage's fate
commitment from multipotent/other cells. Flat variance-explained across all
groups for every factor would mean the grouping isn't picking up meaningful
structure — in which case, per the original plan, fall back to running
everything ungrouped (`groups=False`, single group) as a baseline comparison.

In [ ]:
import mofax as mfx

model = mfx.mofa_model("/home/claude/mofa_unipotency_groups.hdf5")

r2 = model.get_r2()  # columns typically: Factor, View, Group, R2
print(r2.head(20))

r2_pivot = r2.pivot_table(index="Factor", columns=["Group", "View"], values="R2")
r2_pivot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(r2_pivot))))
sns.heatmap(r2_pivot, cmap="viridis", ax=ax, cbar_kws={"label": "R2 (variance explained)"})
ax.set_title("Variance explained per factor, per group x view")
plt.tight_layout()
plt.show()

# quick read: for each factor, which group(s) have disproportionately high R2?
# those are your "this factor is doing something specific to this fate" candidates.
model.close()